# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
!git clone https://github.com/krihna7/flyrank-ml-internship.git /content/flyrank-ML-internship

fatal: destination path '/content/flyrank-ML-internship' already exists and is not an empty directory.


In [26]:
from pathlib import Path

repo = Path("/content/flyrank-ML-internship")

print("Repo exists:", repo.exists())

print("\nNotebooks:")
for p in (repo / "work" / "notebooks").glob("*.ipynb"):
    print(p)

print("\nOutputs:")
output_dir = repo / "work" / "outputs"
if output_dir.exists():
    for p in output_dir.iterdir():
        print(p.name)
else:
    print("Outputs directory does not exist")

Repo exists: True

Notebooks:
/content/flyrank-ML-internship/work/notebooks/w01_research_question.ipynb
/content/flyrank-ML-internship/work/notebooks/w07_action_playbook.ipynb
/content/flyrank-ML-internship/work/notebooks/w03_feature_leakage_check.ipynb
/content/flyrank-ML-internship/work/notebooks/w04_signal_audit.ipynb
/content/flyrank-ML-internship/work/notebooks/w03_data_contract.ipynb
/content/flyrank-ML-internship/work/notebooks/capstone.ipynb
/content/flyrank-ML-internship/work/notebooks/w02_ml_task_framing.ipynb
/content/flyrank-ML-internship/work/notebooks/w04_baseline_score.ipynb
/content/flyrank-ML-internship/work/notebooks/w06_validation_audit.ipynb
/content/flyrank-ML-internship/work/notebooks/w05_model.ipynb

Outputs:
baseline_action_score.csv


In [27]:
from pathlib import Path

notebook_dir = Path("/content/flyrank-ML-internship/work/notebooks")

print("Available notebooks:\n")

for p in sorted(notebook_dir.glob("*.ipynb")):
    print(p.name)

Available notebooks:

capstone.ipynb
w01_research_question.ipynb
w02_ml_task_framing.ipynb
w03_data_contract.ipynb
w03_feature_leakage_check.ipynb
w04_baseline_score.ipynb
w04_signal_audit.ipynb
w05_model.ipynb
w06_validation_audit.ipynb
w07_action_playbook.ipynb


In [28]:
from pathlib import Path

output_dir = Path("/content/flyrank-ML-internship/work/outputs")

print("Outputs directory exists:", output_dir.exists())

if output_dir.exists():
    print("\nFiles:")
    for p in output_dir.iterdir():
        print(p.name)

Outputs directory exists: True

Files:
baseline_action_score.csv


In [29]:
import json
from pathlib import Path

w04_path = Path(
    "/content/flyrank-ML-internship/work/notebooks/w04_baseline_score.ipynb"
)

with open(w04_path, "r", encoding="utf-8") as f:
    w04 = json.load(f)

print("Week-4 notebook cells:", len(w04["cells"]))

for i, cell in enumerate(w04["cells"]):
    print(f"\n{'='*60}")
    print(f"CELL {i} — {cell['cell_type']}")
    print("="*60)
    print("".join(cell["source"])[:4000])

Week-4 notebook cells: 18

CELL 0 — markdown
<a href="https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CELL 1 — markdown
# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

CELL 2 — code
# ML-07 setup: load the starter dataset

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/content/flyrank-ML-internship

In [30]:
# Restore the exact ML-07 baseline for ML-08

from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path("/content/flyrank-ML-internship")
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"
OUTPUT_DIR = REPO / "work/outputs"
OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

# Load the same Week-4 starter dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# --------------------------------------------------
# Exact ML-07 baseline logic
# --------------------------------------------------

baseline = df.copy()

baseline["days_since_last_update"] = pd.to_numeric(
    baseline["days_since_last_update"],
    errors="coerce"
)

baseline["impressions_90d"] = pd.to_numeric(
    baseline["impressions_90d"],
    errors="coerce"
)

baseline = baseline.dropna(
    subset=[
        "content_id",
        "days_since_last_update",
        "impressions_90d"
    ]
).copy()

baseline["days_since_last_update"] = baseline[
    "days_since_last_update"
].clip(lower=0)

baseline["impressions_90d"] = baseline[
    "impressions_90d"
].clip(lower=0)

# 70% visibility
baseline["visibility_score"] = (
    baseline["impressions_90d"].rank(
        method="average",
        pct=True
    )
)

# 30% staleness
baseline["staleness_score"] = (
    baseline["days_since_last_update"].rank(
        method="average",
        pct=True
    )
)

# Final score
baseline["baseline_score"] = (
    0.70 * baseline["visibility_score"]
    + 0.30 * baseline["staleness_score"]
)

# Reason code
baseline["reason_code"] = np.select(
    [
        (baseline["staleness_score"] >= 0.75)
        & (baseline["visibility_score"] >= 0.75),

        baseline["visibility_score"] >= 0.75,

        baseline["staleness_score"] >= 0.75
    ],
    [
        "stale_high_visibility",
        "high_visibility",
        "stale"
    ],
    default="standard_review"
)

# Action
baseline["action"] = "refresh_review"

# Rank
baseline = baseline.sort_values(
    [
        "baseline_score",
        "visibility_score",
        "staleness_score"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Save
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
baseline.to_csv(OUTPUT_PATH, index=False)

print("\n======================================")
print("ML-07 BASELINE RESTORED")
print("======================================")
print("Rows:", len(baseline))
print("Output:", OUTPUT_PATH)
print("Exists:", OUTPUT_PATH.exists())
print("Columns:", len(baseline.columns))

display(
    baseline[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
            "visibility_score",
            "staleness_score"
        ]
    ].head(10)
)

Dataset shape: (30000, 44)

ML-07 BASELINE RESTORED
Rows: 30000
Output: /content/flyrank-ML-internship/work/outputs/baseline_action_score.csv
Exists: True
Columns: 50


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,visibility_score,staleness_score
0,1,content_a5dbb404bdc2,0.991240,stale_high_visibility,refresh_review,106,79035,0.991300,0.991100
1,2,content_cf56e2e2e282,0.989430,stale_high_visibility,refresh_review,194,61678,0.986600,0.996033
2,3,content_7368877ea310,0.989150,stale_high_visibility,refresh_review,194,59472,0.986200,0.996033
3,4,content_47b8b12d581e,0.981067,stale_high_visibility,refresh_review,106,40305,0.976767,0.991100
4,5,content_69fad7e6c50c,0.969470,stale_high_visibility,refresh_review,106,28000,0.960200,0.991100
5,6,content_1bfaa38ff26c,0.967730,stale_high_visibility,refresh_review,194,25715,0.955600,0.996033
6,7,content_482aff19e9cc,0.967183,stale_high_visibility,refresh_review,106,26287,0.956933,0.991100
7,8,content_6ac3ab740bbf,0.961362,stale_high_visibility,refresh_review,106,22462,0.948617,0.991100
8,9,content_ac1d924c6a70,0.959717,stale_high_visibility,refresh_review,106,21853,0.946267,0.991100
9,10,content_cb7e312f5d32,0.959220,stale_high_visibility,refresh_review,151,21272,0.944500,0.993567


In [31]:
print("Baseline rows:", len(baseline))
print("CSV exists:", OUTPUT_PATH.exists())

saved_baseline = pd.read_csv(OUTPUT_PATH)

print("Saved rows:", len(saved_baseline))
print("Missing columns:",
      set(["rank", "content_id", "baseline_score", "reason_code", "action"])
      - set(saved_baseline.columns))

Baseline rows: 30000
CSV exists: True
Saved rows: 30000
Missing columns: set()


In [32]:
import json
from pathlib import Path

path = Path(
    "/content/flyrank-ML-internship/work/notebooks/w02_ml_task_framing.ipynb"
)

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    text = "".join(cell["source"])

    if any(word in text.lower() for word in [
        "target",
        "proxy",
        "label",
        "success metric",
        "metric",
        "outcome"
    ]):
        print("\n" + "=" * 70)
        print(f"CELL {i} — {cell['cell_type']}")
        print("=" * 70)
        print(text)


CELL 0 — markdown
<a href="https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CELL 4 — markdown
## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

CELL 5 — markdown
## Target or proxy

The starter dataset does not contain future content performance, so I use a proxy target.

I define pages with a negative trend (`trend_pct < -10`) as pages that need refreshing.

This label is created using a rule from the available data rather than an observed future outcome.

CELL 7 — markdown
## 3. Success metric

*One metric you can defend. What number means 'good'?*

CELL 8 — markdown
## Success Metric

**Success Metric:** Precision@50

Precision@50 measures how many of the top 50 pages recommended by the model actually need refreshing.

This metric

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I will use a Random Forest model for the SEO Content Prioritization lane.

Random Forest fits this lane because content-prioritization signals can have nonlinear relationships, and several features may interact when identifying pages that need attention. It also provides feature importance, which helps explain which observed signals the model relies on.

The goal is not to maximize model complexity. The model is being used for decision-support, so I will judge it against the Week-4 baseline using the same data, split, and evaluation metric. It will be considered useful only if it provides better measured performance than the baseline.

In [33]:
# ML-08 Section 1 — Verify the repository and Week-4 baseline

from pathlib import Path
import pandas as pd

REPO = Path("/content/flyrank-ML-internship")
BASELINE_PATH = REPO / "work" / "outputs" / "baseline_action_score.csv"

print("Repository exists:", REPO.exists())
print("Baseline path:", BASELINE_PATH)
print("Baseline exists:", BASELINE_PATH.exists())

if BASELINE_PATH.exists():
    baseline_df = pd.read_csv(BASELINE_PATH)

    print("\nBaseline shape:", baseline_df.shape)

    print("\nBaseline columns:")
    print(baseline_df.columns.tolist())

    print("\nFirst 5 rows:")
    display(baseline_df.head())

    print("\nMissing values:")
    print(baseline_df.isna().sum().sort_values(ascending=False).head(10))
else:
    print("\nBaseline file is not in the cloned repository.")
    print("We will restore/recreate it from the Week-4 notebook before Section 3.")

Repository exists: True
Baseline path: /content/flyrank-ML-internship/work/outputs/baseline_action_score.csv
Baseline exists: True

Baseline shape: (30000, 50)

Baseline columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'visibility_score', 'staleness_score', 'baseline_s

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,visibility_score,staleness_score,baseline_score,reason_code,action,rank
0,content_a5dbb404bdc2,client_f369cb89fc,0.0,0.0,LOW,0.0,keyword article,informational,2691.0,17851.0,...,excellent,page_1,stable,3.2,0.991300,0.991100,0.991240,stale_high_visibility,refresh_review,1
1,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,excellent,striking,down,-85.6,0.986600,0.996033,0.989430,stale_high_visibility,refresh_review,2
2,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,excellent,page_3_5,down,-81.5,0.986200,0.996033,0.989150,stale_high_visibility,refresh_review,3
3,content_47b8b12d581e,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,commercial,2713.0,17356.0,...,excellent,page_3_5,down,-63.5,0.976767,0.991100,0.981067,stale_high_visibility,refresh_review,4
4,content_69fad7e6c50c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2902.0,18360.0,...,good,page_1,down,-81.3,0.960200,0.991100,0.969470,stale_high_visibility,refresh_review,5



Missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
cpc                   2468
competition           2468
dtype: int64


In [34]:
# ML-08 Section 2 — Grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

# Load the original dataset
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Check the client identifier available in the starter data
client_candidates = [
    "client_id",
    "client_hash_id"
]

client_column = next(
    (col for col in client_candidates if col in df.columns),
    None
)

print("Client grouping column:", client_column)

assert client_column is not None, (
    "No expected client identifier was found."
)

# Grouped 80/20 split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        groups=df[client_column]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df[client_column].dropna().unique())
test_clients = set(test_df[client_column].dropna().unique())

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", len(train_clients))
print("Test clients:", len(test_clients))

# Critical leakage check
overlap = train_clients.intersection(test_clients)

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0

print("✓ No client appears in both train and test.")
print("✓ Grouped 80/20 split is valid.")

Dataset shape: (30000, 44)
Client grouping column: client_id

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Client overlap: 0
✓ No client appears in both train and test.
✓ Grouped 80/20 split is valid.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a grouped train/test split by client.

The dataset contains multiple content items for the same client, so a random row-level split could place content from the same client in both training and test sets. That could make the measured performance look more optimistic because the model would see client-specific patterns during training.

Grouping by client keeps the clients in the test set unseen during training. This is a more honest test of whether the model can support content-prioritization decisions for clients it did not train on.

I will use an 80/20 grouped split and keep the test clients completely separate from the training clients.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I will train a Random Forest classifier to predict the defined `target_refresh` proxy, where `trend_pct < -10` indicates a page that needs refreshing.

The model will use observed content-performance signals available before the decision. I will exclude identifiers, context fields, the target itself, and fields that directly define the target.

The primary evaluation metric is Precision@50 because the decision is a ranked review queue with limited editorial capacity.

The Random Forest will be compared with the Week-4 baseline on the same held-out test set. The comparison is against the proxy target only; it does not prove that refreshing a page will improve future performance.

In [35]:
# ML-08 Section 3 — Random Forest + Precision@50 comparison

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# --------------------------------------------------
# 1. Prepare the modeling dataset
# --------------------------------------------------

model_df = df.copy()

# Create the ML-03 proxy target
model_df["target_refresh"] = (
    model_df["trend_pct"] < -10
).astype(int)

print("Target distribution:")
print(model_df["target_refresh"].value_counts())

print("\nTarget percentage:")
print(
    (model_df["target_refresh"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

# --------------------------------------------------
# 2. Select allowed features
# --------------------------------------------------

candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "engagement_rate",
    "days_since_last_update"
]

features = [
    col for col in candidate_features
    if col in model_df.columns
]

print("\nFeatures used:")
print(features)

# Keep only required columns
required_columns = features + [
    client_column,
    "content_id",
    "target_refresh"
]

model_df = model_df[required_columns].copy()

# Numeric conversion
for col in features:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

# Remove rows missing target/group
model_df = model_df.dropna(
    subset=["target_refresh", client_column]
).copy()

# Fill feature missing values using training-independent
# median imputation for a simple reproducible baseline.
for col in features:
    model_df[col] = model_df[col].fillna(
        model_df[col].median()
    )

print("\nModeling shape:", model_df.shape)

# --------------------------------------------------
# 3. Grouped train/test split
# --------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df[client_column]
    )
)

train_model = model_df.iloc[train_idx].copy()
test_model = model_df.iloc[test_idx].copy()

print("\nTrain rows:", len(train_model))
print("Test rows:", len(test_model))

print(
    "Train clients:",
    train_model[client_column].nunique()
)

print(
    "Test clients:",
    test_model[client_column].nunique()
)

# Verify no client leakage
train_clients = set(
    train_model[client_column].unique()
)

test_clients = set(
    test_model[client_column].unique()
)

overlap = train_clients.intersection(test_clients)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

# --------------------------------------------------
# 4. Train Random Forest
# --------------------------------------------------

X_train = train_model[features]
y_train = train_model["target_refresh"]

X_test = test_model[features]
y_test = test_model["target_refresh"]

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

# Probability of needing refresh
test_model["model_probability"] = rf.predict_proba(
    X_test
)[:, 1]

# --------------------------------------------------
# 5. Precision@50 helper
# --------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)
    top_k = order[:k]

    return y_true[top_k].mean()

# Model Precision@50
model_p50 = precision_at_k(
    y_test.values,
    test_model["model_probability"].values,
    k=50
)

# --------------------------------------------------
# 6. Calculate Week-4 baseline score
# --------------------------------------------------

baseline_test = df.iloc[test_idx].copy()

baseline_test["visibility_score"] = (
    pd.to_numeric(
        baseline_test["impressions_90d"],
        errors="coerce"
    )
    .rank(method="average", pct=True)
)

baseline_test["staleness_score"] = (
    pd.to_numeric(
        baseline_test["days_since_last_update"],
        errors="coerce"
    )
    .rank(method="average", pct=True)
)

baseline_test["baseline_score"] = (
    0.70 * baseline_test["visibility_score"]
    + 0.30 * baseline_test["staleness_score"]
)

baseline_test["target_refresh"] = (
    baseline_test["trend_pct"] < -10
).astype(int)

baseline_p50 = precision_at_k(
    baseline_test["target_refresh"].values,
    baseline_test["baseline_score"].values,
    k=50
)

# --------------------------------------------------
# 7. Model-vs-baseline table
# --------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "metric": [
        "Precision@50",
        "Precision@50"
    ],
    "score": [
        baseline_p50,
        model_p50
    ]
})

comparison["score_percent"] = (
    comparison["score"] * 100
).round(2)

print("\nMODEL VS BASELINE")
display(comparison)

print(
    f"\nBaseline Precision@50: "
    f"{baseline_p50:.3f}"
)

print(
    f"Random Forest Precision@50: "
    f"{model_p50:.3f}"
)

print(
    f"Difference: "
    f"{model_p50 - baseline_p50:+.3f}"
)

# --------------------------------------------------
# 8. Proxy-target leakage check
# --------------------------------------------------

print("\nPROXY TARGET LEAKAGE CHECK")

print("Target definition:")
print("target_refresh = (trend_pct < -10)")

print("\nModel features:")
for feature in features:
    print("-", feature)

# Check whether trend_pct is directly present as a feature
assert "trend_pct" not in features

print("\n✓ trend_pct itself is not used as a model feature.")

# Check feature correlations with trend_pct
leakage_check = df[
    features + ["trend_pct"]
].copy()

for col in features + ["trend_pct"]:
    leakage_check[col] = pd.to_numeric(
        leakage_check[col],
        errors="coerce"
    )

corr_table = (
    leakage_check.corr(numeric_only=True)["trend_pct"]
    .drop("trend_pct")
    .abs()
    .sort_values(ascending=False)
    .to_frame("absolute_correlation_with_trend_pct")
)

print("\nAbsolute correlations with trend_pct:")
display(corr_table)

Target distribution:
target_refresh
1    18248
0    11752
Name: count, dtype: int64

Target percentage:
target_refresh
1    60.83
0    39.17
Name: proportion, dtype: float64

Features used:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'sessions_90d', 'pageviews_90d', 'users_90d', 'engaged_sessions_90d', 'engagement_rate', 'days_since_last_update']

Modeling shape: (30000, 13)

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0

MODEL VS BASELINE


,method,metric,score,score_percent
0,Week-4 baseline,Precision@50,0.52,52.0
1,Random Forest,Precision@50,0.80,80.0



Baseline Precision@50: 0.520
Random Forest Precision@50: 0.800
Difference: +0.280

PROXY TARGET LEAKAGE CHECK
Target definition:
target_refresh = (trend_pct < -10)

Model features:
- impressions_90d
- clicks_90d
- ctr
- avg_position
- sessions_90d
- pageviews_90d
- users_90d
- engaged_sessions_90d
- engagement_rate
- days_since_last_update

✓ trend_pct itself is not used as a model feature.

Absolute correlations with trend_pct:


,absolute_correlation_with_trend_pct
avg_position,0.047248
impressions_90d,0.024187
days_since_last_update,0.014230
engagement_rate,0.008412
ctr,0.007968
users_90d,0.003704
sessions_90d,0.003162
clicks_90d,0.001778
engaged_sessions_90d,0.001598
pageviews_90d,0.001189


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The Random Forest achieved a Precision@50 of 80.0% on the held-out test set, compared with 52.0% for the Week-4 baseline. This is a measured improvement of 28 percentage points against the defined proxy target.

The held-out test set contained 2,576 true positives, 1,335 false positives, 1,012 false negatives, and 1,240 true negatives. The false positives are pages the model classified as needing refresh even though they did not meet the proxy threshold. The false negatives are pages that met the proxy threshold but were not classified as needing refresh at the selected probability threshold.

Feature importance shows that `impressions_90d` was the strongest feature at 37.77%, followed by `avg_position` at 25.95% and `days_since_last_update` at 10.50%. The model therefore relies most on observed visibility and ranking signals, with content staleness providing additional signal.

The model provides measured improvement over the Week-4 baseline for Precision@50, but this result is against a rule-derived proxy (`trend_pct < -10`), not an observed post-refresh outcome. Therefore, the model should be treated as decision-support for human review rather than proof that refreshing a recommended page will improve future performance.

In [36]:
# ML-08 Section 4 — Feature importance and error analysis

# --------------------------------------------------
# 1. Feature importance
# --------------------------------------------------

feature_importance = (
    pd.DataFrame({
        "feature": features,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("RANDOM FOREST FEATURE IMPORTANCE")
display(feature_importance)

# --------------------------------------------------
# 2. Create predictions
# --------------------------------------------------

test_model["predicted_refresh"] = (
    test_model["model_probability"] >= 0.50
).astype(int)

test_model["actual_refresh"] = (
    test_model["target_refresh"]
)

# --------------------------------------------------
## --------------------------------------------------
# 3. Error categories
# --------------------------------------------------

test_model["error_type"] = np.select(
    [
        (test_model["predicted_refresh"] == 1)
        & (test_model["actual_refresh"] == 1),

        (test_model["predicted_refresh"] == 1)
        & (test_model["actual_refresh"] == 0),

        (test_model["predicted_refresh"] == 0)
        & (test_model["actual_refresh"] == 1),

        (test_model["predicted_refresh"] == 0)
        & (test_model["actual_refresh"] == 0)
    ],
    [
        "true_positive",
        "false_positive",
        "false_negative",
        "true_negative"
    ],
    default="unknown"
)

print("\nERROR COUNTS")

error_counts = (
    test_model["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

display(error_counts)

# --------------------------------------------------
# 4. Top false positives
# --------------------------------------------------

false_positives = (
    test_model[
        test_model["error_type"] == "false_positive"
    ]
    .sort_values("model_probability", ascending=False)
    .head(10)
)

print("\nTOP FALSE POSITIVES")

display(
    false_positives[
        [
            "content_id",
            "model_probability",
            "target_refresh"
        ] + features
    ]
)

# --------------------------------------------------
# 5. Top false negatives
# --------------------------------------------------

false_negatives = (
    test_model[
        test_model["error_type"] == "false_negative"
    ]
    .sort_values("model_probability", ascending=False)
    .head(10)
)

print("\nTOP FALSE NEGATIVES")

display(
    false_negatives[
        [
            "content_id",
            "model_probability",
            "target_refresh"
        ] + features
    ]
)
# --------------------------------------------------
# 4. Top false positives
# --------------------------------------------------

false_positives = (
    test_model[
        test_model["error_type"] == "false_positive"
    ]
    .sort_values("model_probability", ascending=False)
    .head(10)
)

print("\nTOP FALSE POSITIVES")
display(
    false_positives[
        [
            "content_id",
            "model_probability",
            "target_refresh"
        ] + features
    ]
)

# --------------------------------------------------
# 5. Top false negatives
# --------------------------------------------------

false_negatives = (
    test_model[
        test_model["error_type"] == "false_negative"
    ]
    .sort_values("model_probability", ascending=False)
    .head(10)
)

print("\nTOP FALSE NEGATIVES")
display(
    false_negatives[
        [
            "content_id",
            "model_probability",
            "target_refresh"
        ] + features
    ]
)

RANDOM FOREST FEATURE IMPORTANCE


,feature,importance
0,impressions_90d,0.377660
1,avg_position,0.259502
2,days_since_last_update,0.105027
3,ctr,0.068609
4,clicks_90d,0.053430
5,pageviews_90d,0.045625
6,sessions_90d,0.031348
7,users_90d,0.031091
8,engagement_rate,0.015143
9,engaged_sessions_90d,0.012565



ERROR COUNTS


,error_type,count
0,true_positive,2576
1,false_positive,1335
2,true_negative,1240
3,false_negative,1012



TOP FALSE POSITIVES


...

,content_id,model_probability,target_refresh,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,pageviews_90d,users_90d,engaged_sessions_90d,engagement_rate,days_since_last_update
28644,content_7f6b72253c41,0.499492,1,37,0,0.00,2.8,2,1,2,0,0.0,20
5514,content_679828cc0bfe,0.499217,1,27,0,0.00,12.1,1,1,1,0,0.0,20
14440,content_86003ed8aa34,0.499031,1,341,4,1.17,13.3,5,6,5,1,20.0,20
24600,content_bfec24fd6a60,0.498703,1,233,3,1.29,4.6,4,4,4,0,0.0,20
18678,content_d5a9fc0e9ea6,0.498233,1,35,0,0.00,2.7,1,1,1,0,0.0,20
337,content_656372ca5b8d,0.498206,1,28,0,0.00,6.1,4,3,4,0,0.0,20
14808,content_c9e010f15592,0.498152,1,20,0,0.00,8.4,2,2,2,0,0.0,104
10028,content_ca189bed60cb,0.498123,1,46,0,0.00,20.0,1,1,1,0,0.0,20
19969,content_38efc381811c,0.497964,1,11,0,0.00,20.3,7,7,6,0,0.0,102
3729,content_92dfd4fe1208,0.497832,1,31,0,0.00,5.1,1,0,1,0,0.0,20



TOP FALSE POSITIVES


,content_id,model_probability,target_refresh,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,pageviews_90d,users_90d,engaged_sessions_90d,engagement_rate,days_since_last_update
21559,content_bba155c5f227,0.825177,0,1613,1,0.06,3.1,4,5,4,0,0.00,104
27585,content_d831e97bab46,0.788270,0,1360,0,0.00,6.1,15,21,13,0,0.00,104
9192,content_59c260251c82,0.786491,0,8795,2,0.02,34.9,5,6,5,0,0.00,20
22928,content_929aa622b6a0,0.781423,0,11301,3,0.03,2.4,7,7,7,0,0.00,104
18895,content_b50209ec3c8f,0.781290,0,3273,2,0.06,25.8,4,6,3,0,0.00,13
27993,content_26d48a980581,0.778035,0,1266,0,0.00,4.6,5,4,5,0,0.00,106
2552,content_35972508aa52,0.773245,0,8071,3,0.04,23.2,3,3,3,0,0.00,20
15018,content_b60fd33115df,0.765760,0,8027,8,0.10,31.6,7,11,7,1,14.29,14
24534,content_41e79436cdff,0.759471,0,8044,3,0.04,7.9,2,2,2,0,0.00,20
8983,content_cd9dbb3c3105,0.759119,0,1246,1,0.08,3.1,4,3,4,0,0.00,20



TOP FALSE NEGATIVES


,content_id,model_probability,target_refresh,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,pageviews_90d,users_90d,engaged_sessions_90d,engagement_rate,days_since_last_update
28644,content_7f6b72253c41,0.499492,1,37,0,0.00,2.8,2,1,2,0,0.0,20
5514,content_679828cc0bfe,0.499217,1,27,0,0.00,12.1,1,1,1,0,0.0,20
14440,content_86003ed8aa34,0.499031,1,341,4,1.17,13.3,5,6,5,1,20.0,20
24600,content_bfec24fd6a60,0.498703,1,233,3,1.29,4.6,4,4,4,0,0.0,20
18678,content_d5a9fc0e9ea6,0.498233,1,35,0,0.00,2.7,1,1,1,0,0.0,20
337,content_656372ca5b8d,0.498206,1,28,0,0.00,6.1,4,3,4,0,0.0,20
14808,content_c9e010f15592,0.498152,1,20,0,0.00,8.4,2,2,2,0,0.0,104
10028,content_ca189bed60cb,0.498123,1,46,0,0.00,20.0,1,1,1,0,0.0,20
19969,content_38efc381811c,0.497964,1,11,0,0.00,20.3,7,7,6,0,0.0,102
3729,content_92dfd4fe1208,0.497832,1,31,0,0.00,5.1,1,0,1,0,0.0,20


In [37]:
# Final model-vs-baseline summary

final_comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

final_comparison["Precision@50_percent"] = (
    final_comparison["Precision@50"] * 100
).round(2)

improvement_pp = (
    model_p50 - baseline_p50
) * 100

print("FINAL MODEL VS BASELINE")
display(final_comparison)

print(
    f"Measured improvement: {improvement_pp:+.2f} percentage points"
)

FINAL MODEL VS BASELINE


,method,Precision@50,Precision@50_percent
0,Week-4 baseline,0.52,52.0
1,Random Forest,0.80,80.0


Measured improvement: +28.00 percentage points


In [38]:
# Final ML-08 summary

print("FINAL ML-08 RESULTS")
print("=" * 50)

print(f"Week-4 baseline Precision@50 : {baseline_p50:.2%}")
print(f"Random Forest Precision@50  : {model_p50:.2%}")
print(f"Improvement                  : {(model_p50 - baseline_p50):.2%}")

print("\nError counts:")
display(error_counts)

print("\nTop features:")
display(feature_importance.head(5))

FINAL ML-08 RESULTS
Week-4 baseline Precision@50 : 52.00%
Random Forest Precision@50  : 80.00%
Improvement                  : 28.00%

Error counts:


,error_type,count
0,true_positive,2576
1,false_positive,1335
2,true_negative,1240
3,false_negative,1012



Top features:


,feature,importance
0,impressions_90d,0.377660
1,avg_position,0.259502
2,days_since_last_update,0.105027
3,ctr,0.068609
4,clicks_90d,0.053430


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors after the corrected error-analysis cell
- [x] No client names, URLs, or private queries are used
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] The model is compared with the Week-4 baseline on the same held-out test set and Precision@50 metric
- [x] Grouped validation prevents client overlap between training and test data
- [x] `trend_pct` is not directly used as a model feature
- [x] Feature importance and model errors are interpreted
- [ ] Notebook committed to the repository under `work/notebooks/`